# Análisis y Predicción de Riesgo de Ataque al Corazón

Flujo completo de análisis para el dataset `heart_attack_prediction_dataset.csv`:
1. **Carga y Exploración de Datos (EDA)**
2. **Ingeniería de Características (`Blood Pressure` -> `Systolic_BP` y `Diastolic_BP`)**
3. **Definición de Target (`Heart Attack Risk`) y División 70/30**
4. **Preprocesamiento Automático**
5. **Entrenamiento y Validación Cruzada (10 Folds)**
6. **Evaluación en el Conjunto de Test (30%)**
7. **Exportación a `.joblib`**

In [ ]:
import os
import glob
import joblib
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score

In [ ]:
downloads_dir = os.path.expanduser(r'~\Downloads')
pattern = os.path.join(downloads_dir, '*heart*attack*.csv')
files = glob.glob(pattern)

file_path = max(files, key=os.path.getmtime)
print(f'📁 Archivo seleccionado: {file_path}')

df = pd.read_csv(file_path)
df.head()

In [ ]:
# Feature engineering para Blood Pressure
if 'Blood Pressure' in df.columns:
    df[['Systolic_BP', 'Diastolic_BP']] = df['Blood Pressure'].str.split('/', expand=True).astype(float)

predecir = df['Heart Attack Risk']
drop_cols = ['Heart Attack Risk', 'Patient ID', 'Blood Pressure']

X = df.drop(columns=drop_cols)
y = predecir

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

print(f'Muestras totales: {len(df)}')
print(f'Train (70%): {X_train.shape[0]} muestras')
print(f'Test (30%):  {X_test.shape[0]} muestras')

In [ ]:
num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X.select_dtypes(include=['object']).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
    ]
)

In [ ]:
models = {
    'Regresión Logística': LogisticRegression(C=0.1, max_iter=1000, random_state=42),
    'Random Forest Regulado': RandomForestClassifier(n_estimators=100, max_depth=6, min_samples_leaf=5, random_state=42),
    'Árbol de Decisión': DecisionTreeClassifier(max_depth=5, random_state=42)
}

for name, clf in models.items():
    pipe = Pipeline([('prep', preprocessor), ('clf', clf)])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    y_prob = pipe.predict_proba(X_test)[:, 1]
    
    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)
    print(f'MODELO: {name}')
    print(f'Accuracy: {acc:.4f} ({acc*100:.2f}%) | ROC-AUC: {auc:.4f}')
    print(classification_report(y_test, y_pred, target_names=['Bajo Riesgo (0)', 'Alto Riesgo (1)']))
    print(confusion_matrix(y_test, y_pred))
    print('-' * 50)

In [ ]:
best_pipe = Pipeline([('prep', preprocessor), ('clf', LogisticRegression(C=0.1, max_iter=1000, random_state=42))])
best_pipe.fit(X_train, y_train)
joblib.dump(best_pipe, os.path.join(downloads_dir, 'heart_attack_model.joblib'))
print('✅ Modelo guardado como heart_attack_model.joblib')